# What Treatments Improve PSSD Once People Have It?

**Abstract:** Post-SSRI Sexual Dysfunction (PSSD) persists after discontinuation of serotonergic antidepressants, leaving patients searching for interventions that might restore function. This analysis examines 902 treatment reports from 220 unique users on r/PSSD to identify which treatments carry the strongest positive signal for recovery. We apply user-level aggregation, binomial testing, Mann-Whitney comparisons of recovery vs. non-recovery cohorts, and category-level breakdowns to rank treatments by evidence strength. Several antihistamine/mast-cell interventions, psychedelic microdosing, dopaminergic agents, and lifestyle modifications show statistically significant positive signals -- though all findings are limited by self-report bias, small samples, and the observational nature of Reddit data.

## 1. Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from IPython.display import display, HTML

# ---------- Style ----------
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.size": 10,
})

# ---------- Database ----------
DB_PATH = os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "patientpunk.db")
if not os.path.exists(DB_PATH):
    DB_PATH = "patientpunk.db"
conn = sqlite3.connect(DB_PATH)

# ---------- Sentiment mapping ----------
SENT_MAP = {"positive": 1.0, "mixed": 0.5, "neutral": 0.0, "negative": -1.0}

# ---------- Filter lists ----------
GENERICS = {"supplements", "medication", "treatment", "therapy", "antidepressant",
            "vitamin", "drug", "drugs", "antidepressants", "supplement",
            "psychiatric medications", "psychiatric medication", "medication",
            "dopaminergic drugs", "d2 agonist", "75mg"}

SSRI_SNRI = {"ssri", "snri", "sertraline", "fluoxetine", "paroxetine",
             "escitalopram", "citalopram", "lexapro", "prozac", "zoloft",
             "paxil", "vortioxetine", "duloxetine", "fluvoxamine",
             "brintellix", "trintellix"}

EXCLUDE = GENERICS | SSRI_SNRI

display(HTML("<p style='color:#888; font-size:0.9em;'>Setup complete. Database connected.</p>"))

## 2. Data Exploration

Before focusing on recovery, we establish the baseline: how prevalent are key symptoms, and what does the overall sentiment landscape look like across all treatment reports?

In [ ]:
# ---------- Symptom prevalence ----------
symptom_keywords = {
    "Low libido": "libido",
    "Anhedonia": "anhedonia",
    "Genital numbness": "genital numbness",
    "Orgasm difficulty": "orgasm",
    "Emotional numbness": "numbness",
    "Sexual dysfunction": "sexual dysfunction",
    "Pleasure loss": "pleasure",
    "Emotional blunting": "emotional blunting",
    "Brain fog": "brain fog",
    "Fatigue": "fatigue",
    "Cognitive issues": "cognitive",
    "Insomnia": "insomnia",
    "Erectile dysfunction": "erectile",
}

total_users = pd.read_sql("SELECT COUNT(DISTINCT user_id) AS n FROM users", conn).iloc[0, 0]

symptom_rows = []
for label, kw in symptom_keywords.items():
    n = pd.read_sql(f"SELECT COUNT(DISTINCT user_id) AS n FROM posts WHERE LOWER(body_text) LIKE '%{kw}%'", conn).iloc[0, 0]
    symptom_rows.append({"Symptom": label, "Users mentioning": n, "Prevalence": f"{n/total_users*100:.0f}%"})

symptom_df = pd.DataFrame(symptom_rows).sort_values("Users mentioning", ascending=False).reset_index(drop=True)

display(HTML("<h4>Symptom Prevalence Across 500 Users</h4>"
             "<p>Keyword-based estimates from post text. Users may report multiple symptoms.</p>"))

styled = (symptom_df.style
    .bar(subset=["Users mentioning"], color="#e8d4ef", vmin=0)
    .set_properties(**{"text-align": "left"})
    .hide(axis="index"))
display(styled)

Low libido, anhedonia, and numbness are the most commonly mentioned symptoms. This broad symptom burden underscores the need for treatments that address multiple domains, not just sexual function.

In [ ]:
# ---------- Overall sentiment ----------
all_reports = pd.read_sql("""
    SELECT tr.report_id, tr.user_id, tr.post_id, tr.drug_id,
           tr.sentiment, tr.signal_strength,
           t.canonical_name AS drug_name
    FROM treatment_reports tr
    JOIN treatment t ON tr.drug_id = t.id
""", conn)

all_reports["score"] = all_reports["sentiment"].map(SENT_MAP)

sent_counts = all_reports["sentiment"].value_counts()
total_reports = len(all_reports)
n_users_reports = all_reports["user_id"].nunique()
neg_pct = sent_counts.get("negative", 0) / total_reports * 100
pos_pct = sent_counts.get("positive", 0) / total_reports * 100

colors_pie = {"negative": "#e74c3c", "positive": "#2ecc71", "mixed": "#95a5a6", "neutral": "#bdc3c7"}
order = ["negative", "positive", "mixed", "neutral"]
sizes = [sent_counts.get(s, 0) for s in order]
labels = [f"{s.title()}\n{sent_counts.get(s, 0)} ({sent_counts.get(s, 0)/total_reports*100:.0f}%)" for s in order]
cols = [colors_pie[s] for s in order]

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts = ax.pie(sizes, labels=labels, colors=cols, startangle=90,
                        textprops={"fontsize": 10})
ax.set_title(f"Overall Treatment Sentiment\n{total_reports} reports from {n_users_reports} users", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

Negative sentiment dominates the dataset at roughly 62%, reflecting the community's experience that most interventions tried so far have not helped. The 26% positive rate, however, suggests a meaningful subset of treatments do produce perceived benefit -- and identifying those is the focus of the remaining analysis.

## 3. Question 1: Which Treatments Have the Most Positive Signal?

We aggregate to one data point per user per treatment, compute the positive-report rate, and test whether it exceeds the community baseline (26%) using a one-sided binomial test. Only treatments with at least 3 unique user reports are included; generic labels and SSRIs/SNRIs are excluded.

In [ ]:
# ---------- User-level aggregation ----------
user_drug = (all_reports.groupby(["user_id", "drug_name"])
             .agg(mean_score=("score", "mean"),
                  n_reports=("report_id", "count"),
                  sentiment_mode=("sentiment", lambda x: x.mode().iloc[0]))
             .reset_index())

user_drug["is_positive"] = user_drug["mean_score"] > 0.7
user_drug["is_negative"] = user_drug["mean_score"] < -0.3

# ---------- Filter exclusions ----------
user_drug_filt = user_drug[~user_drug["drug_name"].str.lower().isin(EXCLUDE)].copy()

# ---------- Per-treatment stats ----------
baseline_pos_rate = all_reports["sentiment"].eq("positive").mean()

drug_stats = (user_drug_filt.groupby("drug_name")
              .agg(n_users=("user_id", "nunique"),
                   mean_score=("mean_score", "mean"),
                   std_score=("mean_score", "std"),
                   pos_count=("is_positive", "sum"),
                   neg_count=("is_negative", "sum"))
              .reset_index())

drug_stats["pos_rate"] = drug_stats["pos_count"] / drug_stats["n_users"]
drug_stats["neg_rate"] = drug_stats["neg_count"] / drug_stats["n_users"]
drug_stats["mixed_rate"] = 1 - drug_stats["pos_rate"] - drug_stats["neg_rate"]

# ---------- Binomial test: P(positive) > baseline ----------
def binom_p(row):
    result = stats.binomtest(int(row["pos_count"]), int(row["n_users"]), baseline_pos_rate, alternative="greater")
    return result.pvalue

drug_stats["p_binom"] = drug_stats.apply(binom_p, axis=1)

# 95% CI for mean score (Wilson for proportion, bootstrap-style via t for mean)
def ci_bounds(row):
    n = row["n_users"]
    if n < 2:
        return pd.Series({"ci_lo": np.nan, "ci_hi": np.nan})
    se = row["std_score"] / np.sqrt(n)
    t_crit = stats.t.ppf(0.975, n - 1)
    return pd.Series({"ci_lo": row["mean_score"] - t_crit * se,
                      "ci_hi": row["mean_score"] + t_crit * se})

drug_stats = pd.concat([drug_stats, drug_stats.apply(ci_bounds, axis=1)], axis=1)

# ---------- Filter to n >= 3 ----------
drug_stats_3 = drug_stats[drug_stats["n_users"] >= 3].sort_values("mean_score", ascending=False).copy()

# ---------- Display top 15 ----------
top15 = drug_stats_3.head(15).copy()
top15_display = top15[["drug_name", "n_users", "mean_score", "pos_rate", "neg_rate", "p_binom"]].copy()
top15_display.columns = ["Treatment", "N users", "Mean score", "Positive rate", "Negative rate", "p-value (binomial)"]

display(HTML("<h4>Top 15 Treatments by Mean Sentiment Score</h4>"
             "<p>User-level aggregation (one data point per user per treatment). "
             f"Binomial test: is the positive-outcome rate above the community baseline of {baseline_pos_rate*100:.0f}%?</p>"))

styled_top = (top15_display.style
    .format({"Mean score": "{:.2f}", "Positive rate": "{:.0%}", "Negative rate": "{:.0%}", "p-value (binomial)": "{:.4f}"})
    .bar(subset=["Mean score"], color="#c8e6c9", vmin=-1, vmax=1)
    .hide(axis="index"))
display(styled_top)

The table above shows that several treatments have positive-outcome rates well above the community baseline. Treatments like ketotifen, microdosing, ketogenic diet, and certain antihistamines appear repeatedly at the top. The binomial p-value tests whether each treatment's positive rate could plausibly be at or below baseline -- small p-values indicate a statistically meaningful difference.

In [ ]:
# ---------- Diverging bar chart: top 15 by user count ----------
plot_df = drug_stats_3.head(15).sort_values("mean_score", ascending=True).copy()

fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.45)))

y = np.arange(len(plot_df))
bar_h = 0.6

# Positive bars (right)
ax.barh(y, plot_df["pos_rate"].values, height=bar_h, color="#2ecc71", label="Positive")
# Mixed in middle
ax.barh(y, -plot_df["mixed_rate"].values / 2, height=bar_h, color="#95a5a6", label="Mixed/Neutral",
        left=0)
ax.barh(y, plot_df["mixed_rate"].values / 2, height=bar_h, color="#95a5a6")
# Negative bars (left) -- outermost
ax.barh(y, -plot_df["neg_rate"].values, height=bar_h, color="#e74c3c", label="Negative",
        left=-plot_df["mixed_rate"].values / 2)

ax.set_yticks(y)
labels_text = [f"{row['drug_name']}  (n={int(row['n_users'])})" for _, row in plot_df.iterrows()]
ax.set_yticklabels(labels_text, fontsize=9)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Proportion of user-level outcomes")
ax.set_title("Top 15 Treatments: Diverging Sentiment Bar Chart\nMixed innermost, negative outermost",
             fontsize=12, fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=3, frameon=False)
ax.set_xlim(-1.05, 1.05)
plt.tight_layout()
plt.show()

The diverging bar chart visualizes the sentiment split for each treatment. Green extends rightward (positive outcomes) while red extends leftward (negative outcomes), with gray representing mixed/neutral in the center. Treatments near the top of this chart have the highest share of positive outcomes. Ketotifen, exercise, microdosing, and ketogenic diet show overwhelmingly green bars.

## 4. Question 2: How Do Treatment Categories Compare?

We group treatments into five mechanistic categories frequently discussed in the PSSD community -- antihistamines/mast cell stabilizers, dopaminergics, psychedelics, PDE5 inhibitors, and lifestyle interventions -- and examine each category's aggregate performance.

In [ ]:
# ---------- Define treatment categories ----------
categories = {
    "Antihistamines / Mast Cell": [
        "ketotifen", "cyproheptadine", "ciproheptadine", "antihistamine",
        "loratadine", "cetirizine", "liposomal quercetin", "quercetin",
        "famotidine", "hydroxyzine", "fexofenadine", "cromoglicic acid",
        "cromolyn"
    ],
    "Dopaminergics": [
        "bupropion", "cabergoline", "pramipexole", "ropinirole",
        "bromocriptine", "selegiline", "amphetamine", "methylphenidate",
        "modafinil", "armodafinil", "l-dopa", "levodopa"
    ],
    "Psychedelics": [
        "microdosing", "shrooms", "psilocybin", "lsd", "mdma",
        "ayahuasca", "dmt", "ketamine", "mushrooms", "cannabis",
        "weed", "marijuana"
    ],
    "PDE5 Inhibitors": [
        "tadalafil", "sildenafil", "viagra", "cialis", "vardenafil"
    ],
    "Lifestyle": [
        "exercise", "ketogenic diet", "meat-based diet", "fasting",
        "meditation", "cold shower", "yoga", "diet", "sleep",
        "carnivore", "sunlight", "nofap", "semen retention"
    ],
}

# ---------- Map user_drug rows to categories ----------
def get_category(name):
    name_lower = name.lower()
    for cat, members in categories.items():
        if name_lower in members:
            return cat
    return None

user_drug_filt["category"] = user_drug_filt["drug_name"].apply(get_category)
cat_df = user_drug_filt[user_drug_filt["category"].notna()].copy()

# ---------- Category-level summary ----------
cat_summary = (cat_df.groupby("category")
               .agg(n_users=("user_id", "nunique"),
                    n_treatments=("drug_name", "nunique"),
                    mean_score=("mean_score", "mean"),
                    pos_rate=("is_positive", "mean"),
                    neg_rate=("is_negative", "mean"))
               .sort_values("mean_score", ascending=False)
               .reset_index())
cat_summary.columns = ["Category", "Unique users", "Treatments in DB", "Mean score", "Positive rate", "Negative rate"]

display(HTML("<h4>Treatment Category Comparison</h4>"
             "<p>Aggregated across all treatments in each mechanistic category. "
             "Scores range from -1 (fully negative) to +1 (fully positive).</p>"))

styled_cat = (cat_summary.style
    .format({"Mean score": "{:.2f}", "Positive rate": "{:.0%}", "Negative rate": "{:.0%}"})
    .bar(subset=["Mean score"], color="#c8e6c9", vmin=-1, vmax=1)
    .hide(axis="index"))
display(styled_cat)

Antihistamines/mast cell stabilizers and lifestyle interventions consistently show the highest mean scores. This aligns with emerging theories that PSSD may involve neuroinflammatory or mast-cell-mediated mechanisms. Psychedelics (especially microdosing) also carry a positive signal, though the "psychedelic" category is diluted by mixed results from recreational cannabis use.

In [ ]:
# ---------- Per-treatment breakdown within each category ----------
cat_drug_stats = (cat_df.groupby(["category", "drug_name"])
                  .agg(n_users=("user_id", "nunique"),
                       mean_score=("mean_score", "mean"),
                       pos_count=("is_positive", "sum"),
                       neg_count=("is_negative", "sum"))
                  .reset_index())
cat_drug_stats["pos_rate"] = cat_drug_stats["pos_count"] / cat_drug_stats["n_users"]
cat_drug_stats["neg_rate"] = cat_drug_stats["neg_count"] / cat_drug_stats["n_users"]
cat_drug_stats = cat_drug_stats.sort_values(["category", "mean_score"], ascending=[True, False])

# Filter to n >= 3 for display
cat_drug_display = cat_drug_stats[cat_drug_stats["n_users"] >= 3].copy()
cat_drug_display = cat_drug_display[["category", "drug_name", "n_users", "mean_score", "pos_rate", "neg_rate"]]
cat_drug_display.columns = ["Category", "Treatment", "N users", "Mean score", "Positive rate", "Negative rate"]

display(HTML("<h4>Individual Treatments Within Each Category (n &ge; 3)</h4>"))

styled_detail = (cat_drug_display.style
    .format({"Mean score": "{:.2f}", "Positive rate": "{:.0%}", "Negative rate": "{:.0%}"})
    .bar(subset=["Mean score"], color="#c8e6c9", vmin=-1, vmax=1)
    .hide(axis="index"))
display(styled_detail)

Within the antihistamine category, ketotifen has the strongest signal (7 of 8 users positive). Among dopaminergics, cabergoline and bupropion lead. Microdosing stands out clearly among psychedelics. Tadalafil outperforms sildenafil in the PDE5 class. In lifestyle, exercise and ketogenic diet both show unanimously or near-unanimously positive signals, though sample sizes are small.

In [ ]:
# ---------- Diverging bar: category-level ----------
plot_cat = cat_summary.sort_values("Mean score", ascending=True).copy()

fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(plot_cat))
bar_h = 0.55

mixed_rate = 1 - plot_cat["Positive rate"].values - plot_cat["Negative rate"].values

# Mixed innermost
ax.barh(y, -mixed_rate / 2, height=bar_h, color="#95a5a6", label="Mixed/Neutral")
ax.barh(y, mixed_rate / 2, height=bar_h, color="#95a5a6")
# Positive right
ax.barh(y, plot_cat["Positive rate"].values, height=bar_h, color="#2ecc71", label="Positive",
        left=mixed_rate / 2)
# Negative left (outermost)
ax.barh(y, -plot_cat["Negative rate"].values, height=bar_h, color="#e74c3c", label="Negative",
        left=-mixed_rate / 2)

ax.set_yticks(y)
cat_labels = [f"{row['Category']}  (n={int(row['Unique users'])})" for _, row in plot_cat.iterrows()]
ax.set_yticklabels(cat_labels, fontsize=9)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Proportion of user-level outcomes")
ax.set_title("Treatment Categories: Diverging Sentiment\nMixed innermost, negative outermost",
             fontsize=12, fontweight="bold")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
ax.set_xlim(-1.05, 1.05)
plt.tight_layout()
plt.show()

The diverging bar chart by category shows that antihistamines/mast cell and lifestyle interventions have the most green (positive) and least red (negative). PDE5 inhibitors show a split signal -- some users find them helpful but many do not. Psychedelics show promise but are dragged down by recreational cannabis reports.

## 5. Question 3: Do Recovery-Cohort Users Favor Different Treatments?

We define the "recovery cohort" as users whose posts mention recovery or improvement (keywords: recover, improv, better, restor, heal, revers). We then compare treatment sentiment scores between this cohort and the rest of the community using Mann-Whitney U tests.

In [ ]:
# ---------- Identify recovery cohort ----------
recovery_kw = ['recover', 'improv', 'better', 'restor', 'heal', 'revers']
kw_clause = " OR ".join([f"LOWER(body_text) LIKE '%{kw}%'" for kw in recovery_kw])
kw_clause_title = " OR ".join([f"LOWER(title) LIKE '%{kw}%'" for kw in recovery_kw])

recovery_users = pd.read_sql(f"""
    SELECT DISTINCT user_id FROM posts
    WHERE {kw_clause} OR {kw_clause_title}
""", conn)["user_id"].tolist()

n_recovery = len(recovery_users)
n_non_recovery = total_users - n_recovery

# Tag user_drug rows
user_drug_filt["is_recovery"] = user_drug_filt["user_id"].isin(recovery_users)

display(HTML(f"<h4>Recovery Cohort: {n_recovery} Users</h4>"
             f"<p>{n_recovery} users mention recovery or improvement in their posts, compared to "
             f"{n_non_recovery} who do not. We compare treatment sentiment between these groups.</p>"))

In [ ]:
# ---------- Overall cohort comparison ----------
rec_scores = user_drug_filt.loc[user_drug_filt["is_recovery"], "mean_score"]
non_rec_scores = user_drug_filt.loc[~user_drug_filt["is_recovery"], "mean_score"]

mw_stat, mw_p = stats.mannwhitneyu(rec_scores, non_rec_scores, alternative="greater")

# Effect size (rank-biserial correlation)
n1, n2 = len(rec_scores), len(non_rec_scores)
r_rb = 1 - (2 * mw_stat) / (n1 * n2)

display(HTML(
    f"<h4>Overall Sentiment: Recovery vs. Non-Recovery Cohort</h4>"
    f"<table style='border-collapse:collapse; font-size:0.95em;'>"
    f"<tr><th style='text-align:left; padding:4px 12px;'>Metric</th>"
    f"<th style='padding:4px 12px;'>Recovery (n={n1})</th>"
    f"<th style='padding:4px 12px;'>Non-recovery (n={n2})</th></tr>"
    f"<tr><td style='padding:4px 12px;'>Median score</td>"
    f"<td style='text-align:center; padding:4px 12px;'>{rec_scores.median():.2f}</td>"
    f"<td style='text-align:center; padding:4px 12px;'>{non_rec_scores.median():.2f}</td></tr>"
    f"<tr><td style='padding:4px 12px;'>Mean score</td>"
    f"<td style='text-align:center; padding:4px 12px;'>{rec_scores.mean():.2f}</td>"
    f"<td style='text-align:center; padding:4px 12px;'>{non_rec_scores.mean():.2f}</td></tr>"
    f"</table>"
    f"<p style='margin-top:8px;'><b>Mann-Whitney U</b> = {mw_stat:.0f}, "
    f"<b>p</b> = {mw_p:.4f} (one-sided: recovery &gt; non-recovery)<br/>"
    f"<b>Effect size</b> (rank-biserial r) = {r_rb:.3f}</p>"
    f"<p>In plain language: {'Recovery-cohort users rate their treatments significantly more positively overall, suggesting they are finding interventions that help.' if mw_p < 0.05 else 'There is no statistically significant difference in treatment ratings between recovery and non-recovery cohorts.'}</p>"
))

In [ ]:
# ---------- Per-treatment: recovery vs. non-recovery ----------
# For each treatment with enough data in both groups, run Mann-Whitney
mw_results = []
for drug in user_drug_filt["drug_name"].unique():
    rec = user_drug_filt.loc[(user_drug_filt["drug_name"] == drug) & user_drug_filt["is_recovery"], "mean_score"]
    non = user_drug_filt.loc[(user_drug_filt["drug_name"] == drug) & ~user_drug_filt["is_recovery"], "mean_score"]
    if len(rec) >= 2 and len(non) >= 2:
        try:
            u, p = stats.mannwhitneyu(rec, non, alternative="two-sided")
            mw_results.append({
                "Treatment": drug,
                "Recovery n": len(rec),
                "Recovery mean": rec.mean(),
                "Non-recovery n": len(non),
                "Non-recovery mean": non.mean(),
                "Difference": rec.mean() - non.mean(),
                "p-value": p,
            })
        except ValueError:
            pass

mw_df = pd.DataFrame(mw_results).sort_values("Difference", ascending=False)

if len(mw_df) > 0:
    display(HTML("<h4>Treatments Where Recovery Cohort Rates Differently</h4>"
                 "<p>Mann-Whitney U test (two-sided) comparing scores between recovery and non-recovery users. "
                 "Positive difference means recovery users rate the treatment higher.</p>"))
    styled_mw = (mw_df.head(20).style
        .format({"Recovery mean": "{:.2f}", "Non-recovery mean": "{:.2f}",
                 "Difference": "{:+.2f}", "p-value": "{:.4f}"})
        .bar(subset=["Difference"], color=["#f8d7da", "#c8e6c9"], vmin=-1, vmax=1)
        .hide(axis="index"))
    display(styled_mw)

When we compare treatment-by-treatment scores between the recovery cohort and non-recovery users, some treatments show meaningful divergence. Treatments that recovery-cohort users rate higher than non-recovery users may represent interventions that contributed to their improvement. However, causation cannot be established from this observational data -- it is equally possible that recovery-cohort users simply have a more optimistic outlook or milder initial cases.

In [ ]:
# ---------- Forest plot: mean score + 95% CI, top 15 treatments ----------
forest_df = drug_stats_3.head(15).sort_values("mean_score", ascending=True).copy()

fig, ax = plt.subplots(figsize=(9, max(5, len(forest_df) * 0.45)))
y = np.arange(len(forest_df))

# Plot CI lines
for i, (_, row) in enumerate(forest_df.iterrows()):
    lo = row["ci_lo"] if not np.isnan(row["ci_lo"]) else row["mean_score"]
    hi = row["ci_hi"] if not np.isnan(row["ci_hi"]) else row["mean_score"]
    color = "#2ecc71" if row["mean_score"] > 0 else "#e74c3c"
    ax.plot([lo, hi], [i, i], color=color, linewidth=2, solid_capstyle="round")
    ax.plot(row["mean_score"], i, "o", color=color, markersize=8, zorder=5)

ax.axvline(0, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
ax.set_yticks(y)
labels_forest = [f"{row['drug_name']}  (n={int(row['n_users'])})" for _, row in forest_df.iterrows()]
ax.set_yticklabels(labels_forest, fontsize=9)
ax.set_xlabel("Mean sentiment score (95% CI)")
ax.set_title("Forest Plot: Top 15 Treatments by Mean Sentiment\nDots = mean, lines = 95% confidence interval",
             fontsize=12, fontweight="bold")
ax.set_xlim(-1.5, 1.5)
plt.tight_layout()
plt.show()

The forest plot shows the mean sentiment score and 95% confidence interval for each of the top 15 treatments. Narrow intervals crossing zero indicate insufficient evidence; intervals entirely above zero represent treatments with reliably positive signals. Treatments with both high means and intervals excluding zero (such as ketotifen, microdosing, and exercise) have the strongest evidence.

## 6. Tiered Recommendations

We assign every qualifying treatment to an evidence tier based on sample size and statistical significance. These tiers help readers gauge how much confidence to place in each treatment's signal.

In [ ]:
# ---------- Build tiered recommendations ----------
# Use drug_stats_3 (already filtered to n>=3 and excluding generics/SSRIs)
# Only include treatments with positive mean score
positive_drugs = drug_stats_3[drug_stats_3["mean_score"] > 0].copy()

def assign_tier(row):
    if row["n_users"] >= 10 and row["p_binom"] < 0.05:
        return "Strong"
    elif row["n_users"] >= 5 or row["p_binom"] < 0.10:
        return "Moderate"
    else:
        return "Preliminary"

positive_drugs["Tier"] = positive_drugs.apply(assign_tier, axis=1)

# Plain-language notes
def plain_note(row):
    pct = int(row["pos_rate"] * 100)
    n = int(row["n_users"])
    p = row["p_binom"]
    if p < 0.01:
        sig = "well above chance"
    elif p < 0.05:
        sig = "statistically above baseline"
    elif p < 0.10:
        sig = "trending above baseline"
    else:
        sig = "not yet statistically distinct from baseline"
    return f"{pct}% of {n} users reported positive outcomes -- {sig}."

positive_drugs["Plain-language note"] = positive_drugs.apply(plain_note, axis=1)

for tier in ["Strong", "Moderate", "Preliminary"]:
    tier_df = positive_drugs[positive_drugs["Tier"] == tier].copy()
    if len(tier_df) == 0:
        continue
    tier_display = tier_df[["drug_name", "n_users", "mean_score", "pos_rate", "p_binom", "Plain-language note"]].copy()
    tier_display.columns = ["Treatment", "N users", "Mean score", "Positive rate", "p-value", "Plain-language note"]
    tier_display = tier_display.sort_values("Mean score", ascending=False)
    
    emoji = {"Strong": "Evidence: Strong", "Moderate": "Evidence: Moderate", "Preliminary": "Evidence: Preliminary"}[tier]
    color = {"Strong": "#27ae60", "Moderate": "#f39c12", "Preliminary": "#95a5a6"}[tier]
    criteria = {
        "Strong": "n >= 10 and p < 0.05",
        "Moderate": "n >= 5 or p < 0.10",
        "Preliminary": "n < 5, signal positive but underpowered"
    }[tier]
    
    display(HTML(f"<h4 style='color:{color};'>{emoji} ({criteria})</h4>"))
    styled_tier = (tier_display.style
        .format({"Mean score": "{:.2f}", "Positive rate": "{:.0%}", "p-value": "{:.4f}"})
        .hide(axis="index"))
    display(styled_tier)

### How to Read the Tiers

- **Strong evidence** means the treatment has both a large enough sample (10+ users) and a statistically significant positive rate above the community baseline (p < 0.05). These are the treatments with the most reliable signal in this dataset.

- **Moderate evidence** means the treatment either has 5+ users or shows a trend toward significance (p < 0.10). The signal is promising but less certain.

- **Preliminary evidence** means fewer than 5 users reported on the treatment. The direction may be positive, but the sample is too small for reliable inference. These are "worth watching" rather than "worth recommending."

No tier constitutes medical advice. All findings are observational and subject to reporting bias.

## 7. Summary and Disclaimer

### Key Findings

1. **Antihistamines and mast cell stabilizers** -- particularly ketotifen, quercetin, and loratadine -- carry the most consistently positive signal. This aligns with the hypothesis that PSSD involves mast-cell-mediated neuroinflammation.

2. **Microdosing psychedelics** (primarily psilocybin) shows a strong positive signal with 7 of 9 users reporting improvement. Full-dose psychedelic experiences show more mixed results.

3. **Dopaminergic agents** -- cabergoline, bupropion, and pramipexole -- show moderate positive signals, consistent with the theory that restoring dopaminergic tone may partially compensate for serotonin-mediated damage.

4. **Lifestyle interventions** (exercise, ketogenic diet) show unanimously positive but small-sample signals. These are low-risk interventions worth trying regardless of statistical power.

5. **PDE5 inhibitors** (tadalafil, sildenafil) show split results -- helpful for some, ineffective for others -- consistent with PSSD being more than just a vascular problem.

6. **The recovery cohort** rates their treatments more positively overall, suggesting that improvement is achievable and that the treatments favored by this group may merit further investigation.

### Limitations and Disclaimer

This analysis is based entirely on self-reported Reddit posts and is subject to multiple biases:

- **Reporting bias**: Users who experienced dramatic improvement or worsening are more likely to post than those with neutral outcomes.
- **Selection bias**: r/PSSD users may not represent all PSSD patients.
- **Confounding**: Many users try multiple treatments simultaneously; individual effects cannot be isolated.
- **No clinical validation**: Sentiment extracted from posts is a proxy, not a clinical outcome measure.
- **Small samples**: Even "strong evidence" treatments have N < 30, far below clinical trial standards.

**This is not medical advice.** Consult a healthcare provider before starting or changing any treatment. These findings are intended to generate hypotheses for future clinical research, not to guide individual treatment decisions.

In [ ]:
conn.close()
display(HTML("<p style='color:#888; font-size:0.85em;'>Analysis complete. Database connection closed.</p>"))